# 알라딘 도서 페이지 정적 크롤링
알라딘 도서 페이지에서 데이터를 추출하면서 정적 크롤링을 복습합니다.

### 1. 필요한 라이브러리 설치 및 임포트
먼저, 웹 크롤링을 위해 필요한 라이브러리들을 설치하고 임포트합니다.

- bs4: BeautifulSoup 라이브러리는 HTML/XML 페이지를 파싱하여 데이터를 쉽게 추출할 수 있게 도와줍니다.
- requests: HTTP 요청을 보내 웹 페이지의 HTML을 받아오는 라이브러리입니다.
- pandas: 데이터를 표 형태로 처리하고, csv 파일로 저장하는 데 사용됩니다.

In [18]:
!pip install bs4
!pip install requests
!pip install pandas

^C



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 2. HTML 페이지 불러오기 및 파싱
이제 웹 페이지를 불러와서 HTML을 파싱하여 필요한 데이터를 추출하는 작업을 시작합니다.

- requests.get(url): 지정한 URL에 HTTP GET 요청을 보냅니다.
- BeautifulSoup(html, 'html.parser'): 응답 받은 HTML을 BeautifulSoup을 사용해 파싱합니다.

In [3]:
from bs4 import BeautifulSoup
import requests
"""
TODO
1. requests 라이브러리로 url을 받아옵니다.
2. Beautifulsoup로 html 문서를 파싱합니다.
"""

# 알라딘 베스트셀러 페이지 URL
url = "https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=0&page=1&cnt=1000&SortOrder=1"
response = requests.get(url) # 요청 보내기
html = response.text # 응답 받은 HTML 문서
soup = BeautifulSoup(html,"html.parser")# BeautifulSoup으로 파싱
soup


<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">

<html xmlns="http://www.w3.org/1999/xhtml">
<head>
<title>주간 베스트  : 알라딘</title>
<meta content="website" property="og:type"/>
<meta content="베스트셀러 : 알라딘" property="og:title"/>
<meta content="https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1&amp;BestType=Bestseller" property="og:url"/>
<meta content="https://image.aladin.co.kr/img/logo_big.jpg" property="og:image"/>
<meta content="summary" name="twitter:card"/>
<meta content="베스트셀러 : 알라딘" name="twitter:title"/>
<meta content="https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1&amp;BestType=Bestseller" name="twitter:url"/>
<meta content="https://image.aladin.co.kr/img/logo_big.jpg" name="twitter:image"/>
<link href="https://www.aladin.co.kr/shop/common/wbest.aspx?BranchType=1&amp;BestType=Bestseller" rel="canonical"/>
<link href="https://www.aladin.co.kr/m/mbest.aspx?BranchType=1&amp;BestType=Best

### 3. 특정 HTML 요소 선택
크롤링할 HTML 요소를 선택하기 위해 CSS 선택자를 사용하여 데이터를 추출합니다.

- soup.select_one(): CSS 선택자를 사용하여 첫 번째 일치하는 요소를 선택합니다.
- tree: 선택된 HTML 요소(첫 번째 단락)에 대한 정보를 담고 있습니다.

In [ ]:
tree = soup.select_one('.ss_book_list>ul')
tree

<ul>
<li><span class="ss_ht1"><a href="/events/wevent_redirect.aspx?eventid=304414">프로젝트 헤일메리 금속 키링, NASA 공식 키링(앤디 위어 우주 3부작 1권 이상 포함 국내도서 3만 원 이상 구매 시)</a></span><br/></li><li><span class="tit_category">[국내도서]</span> <a class="bo3" href="https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=270454373">프로젝트 헤일메리</a> <span style="color: #777777">ㅣ</span> <a class="ss_f_g2" href="https://www.aladin.co.kr/shop/common/wseriesitem.aspx?SRID=979461">앤디 위어 우주 3부작 </a> </li>
<li><a href="/Search/wSearchResult.aspx?AuthorSearch=앤디+위어@2703260&amp;BranchType=1">앤디 위어</a> (지은이), <a href="/Search/wSearchResult.aspx?AuthorSearch=강동혁@4439327&amp;BranchType=1">강동혁</a> (옮긴이) | <a href="/search/wsearchresult.aspx?PublisherSearch=%ec%95%8c%ec%97%90%ec%9d%b4%ec%b9%98%ec%bd%94%eb%a6%ac%ec%95%84(RHK)@63750&amp;BranchType=1">알에이치코리아(RHK)</a> | 2021년 5월</li><li><span class="">22,000</span>원 → <span class="ss_p2"><em>19,800원</em></span> (<span class="ss_p">10%</span>할인),  마일리지 <span class="ss_p">1,100</span>원</li

### 4. 정보 추출: 제목, 링크, 할인가, 별점
선택한 HTML 요소에서 원하는 데이터를 추출합니다.  
- title_tag.text: title_tag 요소에서 텍스트(제목)를 추출합니다.
- title_tag.attrs['href']: title_tag 요소에서 링크를 추출합니다.
- price_tag.text, review_tag.text : 각각 할인가, 별점을 추출합니다.

In [ ]:
# 제목과 링크 추출
title_tag = tree.select_one(".bo3") 
title_tag.text, title_tag.attrs['href']

('프로젝트 헤일메리', 'https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=270454373')

In [ ]:
# 할인가와 별점 추출
price_tag =tree.select_one(".ss_p2")
review_tag =tree.select_one(".star_score")
price_tag.text, review_tag.text

('19,800원', '9.4')

### 5. 한 페이지에서 모든 도서 정보 추출
한 페이지에 여러 도서가 있을 때, 모든 도서의 정보를 추출합니다.

- soup.select(): 여러 개의 요소를 선택하여 리스트로 반환합니다.
- 각 질문에 대해 for 루프를 돌며 제목, 링크, 할인가, 별점을 추출합니다.

In [11]:
"""
TODO
위 코드를 기반으로 빈칸을 채워 완성하세요.

참조 : try-except문을 통해 원하는 정보가 없는 도서의 경우를 넘어가도록 합니다.
웹사이트의 정보는 늘 균일하지 않기에 크롤링에서 예외처리는 중요합니다.
"""
trees = soup.select(".ss_book_list>ul")
for tree in trees:
    try:
        title = tree.select_one(".bo3")
        title_text = title.text
        title_link = title.attrs['href']
        price = tree.select_one(".ss_p2").text
        review = tree.select_one(".star_score").text
        print(title_text, title_link, price, review)
    except: continue



프로젝트 헤일메리 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=270454373 19,800원 9.4
왕과 사는 남자 각본집 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=388732158 16,200원 10.0
괴테는 모든 것을 말했다 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=376765918 15,300원 8.0
인생을 위한 최소한의 생각 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=385481121 17,820원 9.9
엄마와 딸들의 미친년의 역사 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=388054879 16,020원 10.0
완벽한 원시인 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=386947012 19,800원 9.6
마션 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=270454339 17,820원 9.7
아르테미스 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=270454242 17,820원 8.3
부처님 말씀대로 살아보니 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=385370551 17,010원 9.7
무례한 세상에서 나를 지키는 법 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=388169547 17,820원 10.0
사랑이 있으니 살아집디다 https://www.aladin.co.kr/shop/wproduct.aspx?ItemId=388509769 16,200원 6.4
죽은 왕녀를 위한 파반느 (양장 특별판) https://www.aladin.co.kr/shop/wproduct.a

### 6. 여러 페이지 크롤링
페이지를 변경하면서 여러 페이지의 데이터 크롤링을 해봅시다.

- for page_num in range(1, 4): 1페이지부터 3페이지까지 순차적으로 크롤링합니다.
- 각 페이지에서 데이터를 추출하여 datas 리스트에 추가하고, 이를 pandas DataFrame으로 변환하여 csv 파일로 저장합니다.

In [12]:
import pandas as pd

In [5]:
"""
TODO
1. for loop을 통해 여러 페이지로 이동합니다.
HINT : formatted string을 활용합니다.
2. 내부 for loop에서 위 코드에서 실행했던 한 페이지에서 도서 정보 모으기를 실행합니다.
3. DataFrame에 정보를 저장합니다.
"""
import requests
from bs4 import BeautifulSoup
datas = []
for page_num in range(1, 4):
    url = f"https://www.aladin.co.kr/shop/common/wbest.aspx?BestType=Bestseller&BranchType=1&CID=0&page={page_num}&cnt=1000&SortOrder=1"
    response = requests.get(url)
    html = response.text
    soup = BeautifulSoup(html,'html.parser')
    trees = soup.select(".ss_book_list>ul")
    for tree in trees:
        try:
            title = tree.select_one(".bo3")
            title_text = title.text
            title_link = title.attrs['href']
            price = tree.select_one(".ss_p2").text
            review = tree.select_one(".star_score").text

            datas.append([title_text,title_link,price,review])
        except: continue
import pandas as pd
df = pd.DataFrame(datas,columns=['제목','링크','가격','평점'])
df

,제목,링크,가격,평점
0,프로젝트 헤일메리,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"19,800원",9.4
1,왕과 사는 남자 각본집,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"16,200원",10.0
2,괴테는 모든 것을 말했다,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"15,300원",8.0
3,인생을 위한 최소한의 생각,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"17,820원",9.9
4,엄마와 딸들의 미친년의 역사,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"16,020원",10.0
...,...,...,...,...
123,5번 레인,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"12,600원",9.5
124,"1,000만 원으로 3년 안에 300만 원 월배당 만들기",https://www.aladin.co.kr/shop/wproduct.aspx?It...,"20,700원",9.9
125,필연적 혼자의 시대,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"17,100원",9.6
126,악역 영애 안의 사람 6,https://www.aladin.co.kr/shop/wproduct.aspx?It...,"5,400원",10.0


### 7. 결과 저장
위의 크롤링한 데이터를 csv 파일로 저장합니다.

- df.to_csv(): 추출한 데이터를 csv 파일로 저장합니다. index=False를 설정하여 인덱스를 제외하고 저장합니다.

In [6]:
# csv 파일로 저장해 봅시다.
df.to_csv("aladin_bestseller.csv", index=False)